# **DATASET**

In [ ]:
from openpyxl import load_workbook
import os

FILE_PATH = '/content/DATASET.xlsx'

if os.path.exists(FILE_PATH):
    print(" File found")

    wb = load_workbook(FILE_PATH)
    ws = wb['Dataset_with_color_codings']

    print("Loaded successfully with colors!")

else:
    print("File not found")

✅ File found
✅ Loaded successfully with colors!


In [ ]:
from collections import defaultdict

color_count = defaultdict(int)

for row in ws.iter_rows():
    for cell in row:
        if cell.value:
            color = cell.fill.start_color.rgb
            color_count[color] += 1

print("Color-wise count:\n")

for color, count in sorted(color_count.items(), key=lambda x: x[1], reverse=True):
    print(f"{color} → {count} rows")

🎨 Color-wise count:

FF93C47D → 16541 rows
FFE6B8AF → 9318 rows
FFA4C2F4 → 4236 rows
FFFFD966 → 3424 rows
FFFF00FF → 467 rows
FFA64D79 → 284 rows
FF000000 → 12 rows
FF00FF00 → 12 rows
FF00FFFF → 12 rows


In [ ]:
seen_colors = {}

for row_idx, row in enumerate(ws.iter_rows(), start=1):
    for cell in row:
        if not cell.value:
            continue

        color = cell.fill.start_color.rgb
        value = str(cell.value).strip()

        if color == 'FF000000':
            continue

        if color not in seen_colors:
            seen_colors[color] = (row_idx, value)


print("🎨 One sample row per color:\n")

for color, (row, value) in seen_colors.items():
    print(f"{color} → Row {row} → {value}")

🎨 One sample row per color:

FF00FF00 → Row 2 → 1.0
FF00FFFF → Row 3 → 2.0
FFA64D79 → Row 4 → 3.0
FFFFD966 → Row 5 → 4.0
FFE6B8AF → Row 6 → 5.0
FF93C47D → Row 7 → 6.0
FFA4C2F4 → Row 9 → 8.0
FFFF00FF → Row 733 → 739.0


In [ ]:
color_map = {
    'FF00FF00': 1,
    'FF00FFFF': 2,
    'FFA64D79': 3,
    'FFFFD966': 4,
    'FFE6B8AF': 5,
    'FF93C47D': 6,
    'FFA4C2F4': 7,
    'FFFF00FF': 8
}

In [ ]:
import unicodedata
import re

def deep_clean(text):
    if not isinstance(text, str):
        return text

    text = unicodedata.normalize('NFKC', text)

    hidden_chars = [
        '\xa0', '\u200b', '\u200c', '\u200d',
        '\u200e', '\u200f', '\ufeff', '\u00ad',
        '\u2060', '\u180e', '\t', '\r'
    ]

    for char in hidden_chars:
        text = text.replace(char, '')

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
cleaned_data = []

for row in ws.iter_rows():

    cells = list(row)[2:]

    for cell in cells:
        if not cell.value:
            continue

        value = deep_clean(str(cell.value))
        color = cell.fill.start_color.rgb

        if color == 'FF000000':
            continue

        if value in ['-', '', 'nan']:
            continue

        cleaned_data.append((value, color))

In [ ]:
for v, _ in cleaned_data[:20]:
    print(repr(v))

'AYU'
'vyAdhi-viniScayaH'
'vyādhi-viniścayaḥ'
'व्याधि-विनिश्चयः'
'diagnostic conditions'
'DIS'
'vikAraH'
'vikāraḥ'
'विकारः'
'disease/disorder'
'A'
'doShavaiShamyam'
'dōṣavaiṣamyam'
'दोषवैषम्यम्'
'derangement of dōṣa'
'AA'
'vAtavyAdhiH'
'vātavyādhiḥ'
'वातव्याधिः'
'disorders due to vāta'


In [ ]:
tree = {}
stack = {}

for row in ws.iter_rows(min_row=2):
    cells = list(row)

    NAMC_CODE  = deep_clean(str(cells[2].value)) if cells[2].value else "-"
    NAMC_TERM  = deep_clean(str(cells[3].value)) if cells[3].value else "-"
    NAMC_term_diacritical = deep_clean(str(cells[4].value)) if cells[4].value else "-"
    NAMC_term_DEVANAGARI  = deep_clean(str(cells[5].value)) if cells[5].value else "-"
    Short_definition = deep_clean(str(cells[6].value)) if cells[6].value else "-"
    Long_definition  = deep_clean(str(cells[7].value)) if cells[7].value else "-"
    Ontology_branches = deep_clean(str(cells[8].value)) if cells[8].value else "-"
    english = deep_clean(str(cells[9].value)) if cells[9].value else "-"

    color = cells[2].fill.start_color.rgb

    if color == 'FF000000':
        continue

    level = color_map.get(color)
    if level is None:
        continue

    node = {
        "NAMC_CODE": NAMC_CODE,
        "NAMC_TERM": NAMC_TERM,
        "NAMC_term_diacritical": NAMC_term_diacritical,
        "NAMC_term_DEVANAGARI": NAMC_term_DEVANAGARI,
        "Short_definition": Short_definition,
        "Long_definition": Long_definition,
        "Ontology_branches": Ontology_branches,
        "Name English": english,
        "children": {}
    }

    # 🌱 Level 1
    if level == 1:
        tree[NAMC_CODE] = node
        stack = {1: node["children"]}

    else:
        parent = stack.get(level - 1)
        if parent is None:
            continue

        parent[NAMC_CODE] = node
        stack[level] = node["children"]

        for k in list(stack.keys()):
            if k > level:
                del stack[k]

In [ ]:
import json

print(json.dumps(tree, indent=4, ensure_ascii=False))

output_path = '/content/tree_output.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(tree, f, indent=4, ensure_ascii=False)

print(f"✅ Tree JSON saved at: {output_path}")

Output hidden; open in https://colab.research.google.com to view.